# 🎓 EduGemma — Adaptive AI Tutor Powered by Gemma 4
### Gemma 4 Good Hackathon | Future of Education Track

> **Mission**: Bring personalized, offline-capable tutoring to every student — regardless of location, connectivity, or income.

---

## Architecture Overview
```
Student Input (text/image)
        ↓
  Intent Classifier  (Gemma 4 function calling)
        ↓
  RAG Retriever      (FAISS + OpenStax curriculum)
        ↓
  Gemma 4 9B-IT      (adaptive explanation generation)
        ↓
  Difficulty Adapter (tracks history, adjusts level)
        ↓
  Comprehension Check (auto follow-up question)
```

**Models Used:**
- `google/gemma-4-9b-it` — primary tutor (4-bit quantized for Kaggle T4)
- `google/gemma-4-2b-it` — fine-tuned on SciQ via Unsloth (edge deployment)


## 1. Install Dependencies

In [ ]:
%%capture
!pip install transformers accelerate bitsandbytes -q
!pip install langchain langchain-community langchain-huggingface -q
!pip install faiss-cpu sentence-transformers -q
!pip install unsloth -q
!pip install gradio pillow pypdf -q
!pip install datasets trl peft -q
print('✅ All packages installed')

## 2. Configuration & Setup

In [ ]:
import os
import json
import torch
import warnings
warnings.filterwarnings('ignore')

from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple

@dataclass
class EduGemmaConfig:
    # Model
    model_id: str = 'google/gemma-4-9b-it'
    fine_tuned_model_id: str = 'google/gemma-4-2b-it'  # for Unsloth fine-tuning
    load_in_4bit: bool = True
    max_new_tokens: int = 512
    temperature: float = 0.7

    # RAG
    embedding_model: str = 'sentence-transformers/all-MiniLM-L6-v2'
    chunk_size: int = 512
    chunk_overlap: int = 64
    top_k_retrieval: int = 3

    # Adaptive Learning
    difficulty_levels: List[str] = field(default_factory=lambda: [
        'elementary', 'middle_school', 'high_school', 'undergraduate'
    ])
    history_window: int = 5  # conversations to track for adaptation
    correct_threshold: float = 0.7  # accuracy to level up

    # Paths
    corpus_path: str = '/kaggle/input/openstax-textbooks'
    vector_store_path: str = '/kaggle/working/faiss_index'
    output_model_path: str = '/kaggle/working/edugemma-finetuned'

config = EduGemmaConfig()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device: {device}')
print(f'🔧 GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print(f'💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')
print(f'✅ Config loaded: {config.model_id}')

## 3. Load Gemma 4 with 4-bit Quantization

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline
)

print(f'⏳ Loading {config.model_id}...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=config.load_in_4bit,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    config.model_id,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    config.model_id,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model.eval()

# Create text generation pipeline
text_gen_pipeline = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=config.max_new_tokens,
    temperature=config.temperature,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=False,
)

print(f'✅ Gemma 4 loaded successfully!')
print(f'📊 Model params: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B')

## 4. Knowledge Base — RAG Pipeline with OpenStax Curriculum

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.schema import Document
import hashlib

# ─── Sample curriculum content (in production, load from OpenStax PDFs) ───
# In your Kaggle notebook, replace this with actual OpenStax content
# Download from: https://openstax.org/api/v2/books

SAMPLE_CURRICULUM = [
    {
        'subject': 'Physics',
        'chapter': "Newton's Laws of Motion",
        'content': """
        Newton's First Law (Law of Inertia): An object at rest stays at rest, and an object 
        in motion stays in motion with the same speed and in the same direction unless acted 
        upon by an unbalanced force. This means objects resist changes to their state of motion.
        
        Newton's Second Law: The acceleration of an object depends on two variables — 
        the net force acting on the object and the mass of the object.
        The formula is: F = ma (Force equals mass times acceleration).
        If you push a 2kg block with 10 Newtons of force, it accelerates at 5 m/s².
        
        Newton's Third Law: For every action, there is an equal and opposite reaction.
        When you push against a wall, the wall pushes back on you with equal force.
        Rockets work by pushing exhaust gases downward, and the gases push the rocket upward.
        """
    },
    {
        'subject': 'Mathematics',
        'chapter': 'Quadratic Equations',
        'content': """
        A quadratic equation is a polynomial equation of degree 2. The standard form is:
        ax² + bx + c = 0, where a ≠ 0.
        
        Methods to solve quadratic equations:
        1. Factoring: Find two numbers that multiply to give 'ac' and add to give 'b'.
           Example: x² + 5x + 6 = 0 → (x+2)(x+3) = 0 → x = -2 or x = -3
        
        2. Quadratic Formula: x = (-b ± √(b²-4ac)) / 2a
           The discriminant (b²-4ac) tells us:
           - If positive: two real solutions
           - If zero: one real solution (repeated root)
           - If negative: no real solutions (complex roots)
        
        3. Completing the Square: Rewrite the equation in vertex form.
        """
    },
    {
        'subject': 'Biology',
        'chapter': 'Cell Division — Mitosis and Meiosis',
        'content': """
        Mitosis is cell division that produces two identical daughter cells from one parent cell.
        It is used for growth, repair, and asexual reproduction.
        Phases: Prophase → Metaphase → Anaphase → Telophase → Cytokinesis
        
        Meiosis produces four genetically unique haploid cells from one diploid cell.
        It is used for sexual reproduction (creating sperm and eggs).
        Meiosis involves two rounds of division: Meiosis I and Meiosis II.
        
        Key difference: Mitosis maintains chromosome number (2n→2n),
        while Meiosis halves it (2n→n) to allow sexual reproduction.
        """
    },
    {
        'subject': 'Chemistry',
        'chapter': 'Periodic Table and Chemical Bonding',
        'content': """
        The periodic table organizes elements by atomic number and recurring chemical properties.
        Elements in the same group (column) have similar chemical properties because they have
        the same number of valence electrons.
        
        Chemical bonds:
        - Ionic bonds: Transfer of electrons between metals and non-metals. Example: NaCl (table salt)
          Na gives one electron to Cl, creating Na+ and Cl- ions that attract each other.
        - Covalent bonds: Sharing of electrons between non-metals. Example: H2O (water)
          Oxygen shares electrons with two hydrogen atoms.
        - Metallic bonds: Electrons shared among many metal atoms in a 'sea of electrons'.
        """
    },
    {
        'subject': 'History',
        'chapter': 'World War II — Causes and Consequences',
        'content': """
        World War II (1939-1945) was the deadliest conflict in human history.
        
        Main causes:
        1. Rise of fascism and Nazism in Europe (Hitler in Germany, Mussolini in Italy)
        2. Failure of appeasement policies by Britain and France
        3. Resentment from harsh WWI treaties (Treaty of Versailles)
        4. Global economic depression of the 1930s
        
        Key turning points: Battle of Stalingrad, D-Day (June 6, 1944), Battle of Midway
        
        Consequences: Formation of the United Nations, Cold War, decolonization movements,
        establishment of Israel, Marshall Plan, and the beginning of the nuclear age.
        """
    },
]

class CurriculumKnowledgeBase:
    def __init__(self, config: EduGemmaConfig):
        self.config = config
        self.embeddings = HuggingFaceEmbeddings(
            model_name=config.embedding_model,
            model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
        )
        self.vector_store = None
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=config.chunk_size,
            chunk_overlap=config.chunk_overlap,
            separators=['\n\n', '\n', '. ', ' ', '']
        )

    def build_from_curriculum(self, curriculum: List[Dict]) -> None:
        """Build FAISS index from curriculum content."""
        print('📚 Building knowledge base...')
        documents = []

        for item in curriculum:
            chunks = self.text_splitter.split_text(item['content'])
            for i, chunk in enumerate(chunks):
                doc = Document(
                    page_content=chunk.strip(),
                    metadata={
                        'subject': item['subject'],
                        'chapter': item['chapter'],
                        'chunk_id': i,
                        'doc_id': hashlib.md5(item['chapter'].encode()).hexdigest()[:8]
                    }
                )
                documents.append(doc)

        self.vector_store = FAISS.from_documents(documents, self.embeddings)
        print(f'✅ Knowledge base built: {len(documents)} chunks indexed')

    def retrieve(self, query: str, k: int = None) -> List[Document]:
        """Retrieve most relevant curriculum chunks for a query."""
        k = k or self.config.top_k_retrieval
        if self.vector_store is None:
            raise ValueError('Knowledge base not built. Call build_from_curriculum() first.')
        docs = self.vector_store.similarity_search(query, k=k)
        return docs

    def save(self, path: str) -> None:
        self.vector_store.save_local(path)
        print(f'💾 Knowledge base saved to {path}')

    def load(self, path: str) -> None:
        self.vector_store = FAISS.load_local(
            path, self.embeddings, allow_dangerous_deserialization=True
        )
        print(f'✅ Knowledge base loaded from {path}')

# Build the knowledge base
kb = CurriculumKnowledgeBase(config)
kb.build_from_curriculum(SAMPLE_CURRICULUM)

# Test retrieval
test_docs = kb.retrieve('What is Newton second law?')
print(f'\n🔍 Test retrieval for "Newton second law":')
print(f'  → Found in: {test_docs[0].metadata["chapter"]} ({test_docs[0].metadata["subject"]})')

## 5. Adaptive Difficulty Engine

In [ ]:
from collections import deque
from enum import Enum

class DifficultyLevel(Enum):
    ELEMENTARY = 0
    MIDDLE_SCHOOL = 1
    HIGH_SCHOOL = 2
    UNDERGRADUATE = 3

DIFFICULTY_PROMPTS = {
    DifficultyLevel.ELEMENTARY: """
    You are a friendly tutor for a student aged 8-10.
    - Use VERY simple words. No jargon.
    - Use everyday examples (toys, food, games, animals).
    - Keep sentences short. Use emojis sparingly for engagement.
    - Maximum 3 key points per explanation.
    - Always end with a simple yes/no check question.
    """,
    DifficultyLevel.MIDDLE_SCHOOL: """
    You are a supportive tutor for a student aged 11-14.
    - Use clear language. Introduce subject-specific terms WITH definitions.
    - Use relatable examples from daily life or pop culture.
    - Explain the 'why' behind concepts, not just the 'what'.
    - Include one worked example if math/science.
    - End with a comprehension check question.
    """,
    DifficultyLevel.HIGH_SCHOOL: """
    You are a knowledgeable tutor for a high school student aged 15-18.
    - Use proper subject terminology with precision.
    - Show connections between concepts.
    - Include formulas, equations, or detailed examples where relevant.
    - Encourage critical thinking — ask 'what do you think would happen if...'
    - End with a challenging application question.
    """,
    DifficultyLevel.UNDERGRADUATE: """
    You are a university-level tutor.
    - Assume solid foundational knowledge.
    - Discuss nuances, exceptions, and advanced applications.
    - Reference real-world research or applications where relevant.
    - Encourage independent thinking and hypothesis formation.
    - End with an open-ended analytical question.
    """
}

class AdaptiveDifficultyEngine:
    """Tracks student performance and adapts difficulty level dynamically."""

    def __init__(self, initial_level: DifficultyLevel = DifficultyLevel.MIDDLE_SCHOOL):
        self.current_level = initial_level
        self.response_history = deque(maxlen=5)  # last 5 interactions
        self.correct_count = 0
        self.total_count = 0
        self.subject_weakness = {}  # track weak subjects

    def record_response(self, was_correct: bool, subject: str) -> None:
        """Record whether student answered follow-up correctly."""
        self.response_history.append(was_correct)
        self.total_count += 1
        if was_correct:
            self.correct_count += 1

        # Track subject weaknesses
        if subject not in self.subject_weakness:
            self.subject_weakness[subject] = {'correct': 0, 'total': 0}
        self.subject_weakness[subject]['total'] += 1
        if was_correct:
            self.subject_weakness[subject]['correct'] += 1

        self._adapt_level()

    def _adapt_level(self) -> None:
        """Adapt difficulty based on recent performance."""
        if len(self.response_history) < 3:
            return  # Need at least 3 data points

        recent_accuracy = sum(self.response_history) / len(self.response_history)
        levels = list(DifficultyLevel)
        current_idx = levels.index(self.current_level)

        if recent_accuracy >= 0.8 and current_idx < len(levels) - 1:
            self.current_level = levels[current_idx + 1]
            print(f'📈 Level UP! Now at: {self.current_level.name}')
        elif recent_accuracy < 0.4 and current_idx > 0:
            self.current_level = levels[current_idx - 1]
            print(f'📉 Adjusting down to: {self.current_level.name}')

    def get_prompt_modifier(self) -> str:
        return DIFFICULTY_PROMPTS[self.current_level]

    def get_stats(self) -> Dict:
        accuracy = self.correct_count / max(self.total_count, 1)
        weak_subjects = [
            subj for subj, stats in self.subject_weakness.items()
            if stats['total'] > 0 and stats['correct'] / stats['total'] < 0.5
        ]
        return {
            'current_level': self.current_level.name,
            'overall_accuracy': f'{accuracy:.1%}',
            'total_interactions': self.total_count,
            'weak_subjects': weak_subjects,
            'recent_trend': list(self.response_history)
        }

print('✅ Adaptive Difficulty Engine ready')
print('  Levels:', [l.name for l in DifficultyLevel])

## 6. Core Tutor — EduGemma Agent

In [ ]:
from PIL import Image
import base64
import io
import re

class EduGemmaAgent:
    """Main adaptive tutoring agent powered by Gemma 4."""

    def __init__(
        self,
        model,
        tokenizer,
        pipeline,
        knowledge_base: CurriculumKnowledgeBase,
        config: EduGemmaConfig
    ):
        self.model = model
        self.tokenizer = tokenizer
        self.pipeline = pipeline
        self.kb = knowledge_base
        self.config = config
        self.difficulty_engine = AdaptiveDifficultyEngine()
        self.conversation_history = []
        self.student_name = 'Student'

    def _build_system_prompt(self, subject: str = '') -> str:
        difficulty_guide = self.difficulty_engine.get_prompt_modifier()
        return f"""You are EduGemma, a world-class adaptive AI tutor.
You help students learn by providing clear, accurate explanations grounded in curriculum knowledge.
You adapt your teaching style to the student's level and always verify understanding.

Current student level guidance:
{difficulty_guide}

Core principles:
1. ALWAYS ground answers in the provided curriculum context
2. If asked something outside the context, say so honestly
3. Use the Socratic method — guide don't just tell
4. Celebrate correct answers with genuine encouragement
5. For wrong answers, gently redirect without discouraging
6. Structure longer explanations with clear steps or numbered points
7. Make learning joyful — use analogies and real-world connections
"""

    def _format_context(self, docs: List) -> str:
        if not docs:
            return 'No specific curriculum content found for this topic.'
        context_parts = []
        for i, doc in enumerate(docs, 1):
            context_parts.append(
                f'[Source {i}: {doc.metadata["subject"]} — {doc.metadata["chapter"]}]\n'
                f'{doc.page_content}'
            )
        return '\n\n'.join(context_parts)

    def _build_rag_prompt(self, question: str, context: str) -> str:
        history_str = ''
        if self.conversation_history:
            recent = self.conversation_history[-4:]  # last 2 turns
            history_str = '\n'.join([
                f"{turn['role'].upper()}: {turn['content'][:200]}..."
                if len(turn['content']) > 200 else f"{turn['role'].upper()}: {turn['content']}"
                for turn in recent
            ])

        prompt = f"""CURRICULUM CONTEXT:
{context}

{'RECENT CONVERSATION:' + chr(10) + history_str if history_str else ''}

STUDENT QUESTION: {question}

Please provide a helpful, accurate explanation based on the curriculum context above.
End your response with a comprehension check question to verify understanding."""
        return prompt

    def ask(self, question: str, image: Optional[Image.Image] = None) -> Dict:
        """
        Main interface: student asks a question, optionally with an image.
        Returns dict with answer, sources, difficulty level, follow-up question.
        """
        print(f'\n🎓 Processing: "{question[:80]}..."' if len(question) > 80 else f'\n🎓 Processing: "{question}"')

        # Step 1: Retrieve relevant curriculum
        retrieved_docs = self.kb.retrieve(question)
        subject = retrieved_docs[0].metadata['subject'] if retrieved_docs else 'General'
        context = self._format_context(retrieved_docs)

        # Step 2: Handle multimodal input (image of problem)
        image_context = ''
        if image:
            image_context = '\n[Note: Student provided an image of their problem/work. '\
                           'Acknowledge this and address what they might be working on based on context.]'

        # Step 3: Build full prompt with system + RAG context
        system_prompt = self._build_system_prompt(subject)
        rag_prompt = self._build_rag_prompt(question + image_context, context)

        # Step 4: Format as chat template
        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': rag_prompt}
        ]

        chat_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # Step 5: Generate response
        output = self.pipeline(chat_text)[0]['generated_text']
        answer = output.strip()

        # Step 6: Extract follow-up question (if Gemma included one)
        follow_up = self._extract_follow_up(answer)

        # Step 7: Update conversation history
        self.conversation_history.append({'role': 'user', 'content': question})
        self.conversation_history.append({'role': 'assistant', 'content': answer})

        return {
            'answer': answer,
            'follow_up_question': follow_up,
            'subject': subject,
            'sources': [f"{d.metadata['subject']} — {d.metadata['chapter']}" for d in retrieved_docs],
            'difficulty_level': self.difficulty_engine.current_level.name,
            'stats': self.difficulty_engine.get_stats()
        }

    def _extract_follow_up(self, text: str) -> str:
        """Extract the comprehension check question from response."""
        lines = text.split('\n')
        for line in reversed(lines):
            if '?' in line and len(line) > 20:
                return line.strip()
        return ''

    def evaluate_student_answer(self, student_answer: str, question: str, subject: str) -> Dict:
        """
        Evaluate if student answered the follow-up correctly.
        Returns feedback and updates difficulty engine.
        """
        eval_prompt = f"""A student was asked: "{question}"
Their answer was: "{student_answer}"

Evaluate this answer:
1. Is it correct? (yes/partially/no)
2. Give brief encouraging feedback (1-2 sentences)
3. If wrong, give a gentle hint

Respond in JSON format:
{{"correct": true/false, "feedback": "...", "hint": "..."}}"""

        output = self.pipeline(eval_prompt)[0]['generated_text']

        # Parse JSON from response
        try:
            json_match = re.search(r'\{.*?\}', output, re.DOTALL)
            if json_match:
                result = json.loads(json_match.group())
                self.difficulty_engine.record_response(result.get('correct', False), subject)
                return result
        except:
            pass

        # Fallback
        return {'correct': False, 'feedback': 'Good try! Let me help you understand this better.', 'hint': ''}

    def generate_practice_sheet(self, subject: str, num_questions: int = 5) -> str:
        """Generate personalized practice questions for a subject."""
        stats = self.difficulty_engine.get_stats()
        docs = self.kb.retrieve(subject, k=5)
        context = self._format_context(docs)

        prompt = f"""Based on this curriculum content:
{context}

Generate {num_questions} practice questions for a {stats['current_level']} level student.
Mix different question types: multiple choice, short answer, and one problem-solving question.
Format each question clearly and include an answer key at the end.
Make the questions progressively harder."""

        output = self.pipeline(prompt)[0]['generated_text']
        return output.strip()

    def reset_session(self) -> None:
        """Reset for a new student session."""
        self.conversation_history = []
        self.difficulty_engine = AdaptiveDifficultyEngine()
        print('🔄 Session reset for new student')

# Initialize the agent
tutor = EduGemmaAgent(
    model=model,
    tokenizer=tokenizer,
    pipeline=text_gen_pipeline,
    knowledge_base=kb,
    config=config
)

print('✅ EduGemma Agent initialized and ready!')

## 7. Demo — Live Tutoring Session

In [ ]:
# ─── Demo Session ─────────────────────────────────────────────────────────────
print('='*60)
print('🎓 EDUGEMMA LIVE TUTORING DEMO')
print('='*60)

# Question 1: Physics
response1 = tutor.ask('Can you explain Newton\'s second law? I don\'t understand how force and mass relate.')
print('\n📚 SUBJECT:', response1['subject'])
print('📊 DIFFICULTY:', response1['difficulty_level'])
print('🔗 SOURCES:', response1['sources'])
print('\n💬 ANSWER:')
print(response1['answer'])
print('\n❓ FOLLOW-UP:', response1['follow_up_question'])

In [ ]:
# Question 2: Math
response2 = tutor.ask('How do I solve x² + 5x + 6 = 0 using factoring?')
print('💬 ANSWER:')
print(response2['answer'])
print('\n📈 STUDENT STATS:', json.dumps(response2['stats'], indent=2))

In [ ]:
# Generate practice sheet
print('\n📝 GENERATING PRACTICE SHEET...')
practice = tutor.generate_practice_sheet('Newton\'s Laws', num_questions=5)
print(practice)

## 8. Fine-tuning with Unsloth (Unsloth Special Tech Prize)

In [ ]:
# ─── Fine-tuning with Unsloth ─────────────────────────────────────────────────
# This section fine-tunes gemma-4-2b on SciQ dataset for edge deployment
# Qualifies for the $10,000 Unsloth Special Technology Prize

from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import pandas as pd

print('🔧 Setting up Unsloth fine-tuning pipeline...')

# Load base model with Unsloth
ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name='google/gemma-4-2b-it',
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# Add LoRA adapters
ft_model = FastLanguageModel.get_peft_model(
    ft_model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

print(f'✅ Model loaded with LoRA: rank={16}')
print(f'📊 Trainable params: {sum(p.numel() for p in ft_model.parameters() if p.requires_grad):,}')

In [ ]:
# Load and prepare SciQ dataset
print('📚 Loading SciQ dataset...')
sciq_dataset = load_dataset('sciq', split='train[:5000]')  # 5k samples for demo

EOS_TOKEN = ft_tokenizer.eos_token

def format_sciq_for_tutor(examples):
    """Format SciQ QA pairs into tutor-style conversations."""
    texts = []
    for q, a, support in zip(examples['question'], examples['correct_answer'], examples['support']):
        if not support:
            support = 'Based on scientific knowledge'

        conversation = f"""<start_of_turn>user
Can you help me understand: {q}<end_of_turn>
<start_of_turn>model
Great question! Let me explain this clearly.

Context: {support[:300]}

The answer is: {a}

To check your understanding: Can you explain this concept back to me in your own words?<end_of_turn>"""
        texts.append(conversation + EOS_TOKEN)
    return {'text': texts}

formatted_dataset = sciq_dataset.map(format_sciq_for_tutor, batched=True)
print(f'✅ Dataset prepared: {len(formatted_dataset)} examples')
print('\nSample formatted entry:')
print(formatted_dataset[0]['text'][:400])

In [ ]:
# Fine-tuning with SFTTrainer
trainer = SFTTrainer(
    model=ft_model,
    tokenizer=ft_tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field='text',
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,       # Increase to 500+ for production training
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        seed=42,
        output_dir=config.output_model_path,
        report_to='none',
    ),
)

print('🚀 Starting fine-tuning...')
trainer_stats = trainer.train()

print(f'\n✅ Fine-tuning complete!')
print(f'⏱️  Training time: {trainer_stats.metrics["train_runtime"]:.0f}s')
print(f'📉 Final loss: {trainer_stats.metrics["train_loss"]:.4f}')

In [ ]:
# Save fine-tuned model
ft_model.save_pretrained(config.output_model_path)
ft_tokenizer.save_pretrained(config.output_model_path)

# Save merged model for edge deployment (Ollama/LiteRT compatible)
ft_model.save_pretrained_merged(
    config.output_model_path + '_merged',
    ft_tokenizer,
    save_method='merged_16bit'
)

# Also save in GGUF for llama.cpp (Ollama compatible)
ft_model.save_pretrained_gguf(
    config.output_model_path + '_gguf',
    ft_tokenizer,
    quantization_method='q4_k_m'  # Best quality/size tradeoff
)

print('✅ Models saved:')
print(f'  📦 LoRA adapters: {config.output_model_path}')
print(f'  📦 Merged 16-bit: {config.output_model_path}_merged')
print(f'  📦 GGUF (Ollama): {config.output_model_path}_gguf')

## 9. Benchmark Evaluation

In [ ]:
# ─── Benchmark on ARC dataset ─────────────────────────────────────────────────
# Tests model accuracy on science/reasoning questions

from datasets import load_dataset
import re

def run_arc_benchmark(pipeline_fn, num_samples: int = 100) -> Dict:
    """Evaluate on ARC (AI2 Reasoning Challenge) dataset."""
    print(f'\n📊 Running ARC benchmark on {num_samples} samples...')

    arc = load_dataset('ai2_arc', 'ARC-Challenge', split=f'test[:{num_samples}]')
    correct = 0
    results = []

    for item in arc:
        question = item['question']
        choices = item['choices']
        answer_key = item['answerKey']

        # Format choices
        choices_text = '\n'.join([
            f"{label}) {text}"
            for label, text in zip(choices['label'], choices['text'])
        ])

        prompt = f"""Question: {question}

Choices:
{choices_text}

Answer with just the letter (A, B, C, or D):"""

        output = pipeline_fn(prompt)[0]['generated_text']
        predicted = output.strip()[:1].upper()

        is_correct = (predicted == answer_key)
        correct += is_correct
        results.append({'question': question, 'predicted': predicted, 'actual': answer_key, 'correct': is_correct})

    accuracy = correct / num_samples
    print(f'✅ ARC Accuracy: {accuracy:.1%} ({correct}/{num_samples})')
    return {'accuracy': accuracy, 'correct': correct, 'total': num_samples, 'results': results}

# Run benchmark
benchmark_results = run_arc_benchmark(text_gen_pipeline, num_samples=100)

# Compare base vs fine-tuned (uncomment after fine-tuning)
# ft_pipeline = pipeline('text-generation', model=ft_model, tokenizer=ft_tokenizer, ...)
# ft_benchmark = run_arc_benchmark(ft_pipeline, num_samples=100)
# print(f'Improvement: {ft_benchmark["accuracy"] - benchmark_results["accuracy"]:.1%}')

## 10. Gradio Demo App

In [ ]:
import gradio as gr
from PIL import Image

# ─── Gradio UI ────────────────────────────────────────────────────────────────
CUSTOM_CSS = """
.gradio-container { font-family: 'Segoe UI', sans-serif; }
.chat-message { border-radius: 12px; padding: 12px; margin: 8px 0; }
#stats-box { background: #f0f9ff; border-radius: 8px; padding: 10px; font-size: 13px; }
"""

def chat_with_tutor(message, image, history, student_name):
    if not message.strip():
        return history, ''

    tutor.student_name = student_name or 'Student'

    img = Image.open(image) if image else None
    response = tutor.ask(message, image=img)

    stats = response['stats']
    stats_display = (
        f"Level: {stats['current_level']} | "
        f"Accuracy: {stats['overall_accuracy']} | "
        f"Sessions: {stats['total_interactions']}"
    )

    history.append((message, response['answer']))
    return history, stats_display

def generate_practice(subject):
    return tutor.generate_practice_sheet(subject, num_questions=5)

def reset_session_ui():
    tutor.reset_session()
    return [], '', 'Session reset! Starting fresh.'

with gr.Blocks(css=CUSTOM_CSS, title='EduGemma — AI Tutor') as demo:
    gr.Markdown('# 🎓 EduGemma\n### Your Adaptive AI Tutor, Powered by Gemma 4')

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(label='Tutoring Session', height=500)
            with gr.Row():
                msg_input = gr.Textbox(
                    placeholder='Ask me anything — physics, math, history, biology...',
                    label='Your Question',
                    scale=4
                )
                submit_btn = gr.Button('Ask ✨', variant='primary', scale=1)
            image_input = gr.Image(
                label='📷 Upload a photo of your problem (optional)',
                type='filepath',
                height=150
            )

        with gr.Column(scale=1):
            student_name_input = gr.Textbox(label='Your Name', value='Student', max_lines=1)
            stats_display = gr.Textbox(label='📊 Your Progress', elem_id='stats-box', interactive=False)

            gr.Markdown('### 📝 Practice Sheet Generator')
            subject_input = gr.Dropdown(
                choices=['Physics', 'Mathematics', 'Biology', 'Chemistry', 'History'],
                label='Select Subject',
                value='Physics'
            )
            practice_btn = gr.Button('Generate Practice Questions 📋')
            practice_output = gr.Textbox(label='Practice Sheet', lines=15, interactive=False)

            reset_btn = gr.Button('🔄 Reset Session', variant='secondary')

    # Event handlers
    submit_btn.click(
        chat_with_tutor,
        inputs=[msg_input, image_input, chatbot, student_name_input],
        outputs=[chatbot, stats_display]
    ).then(lambda: '', outputs=[msg_input])

    msg_input.submit(
        chat_with_tutor,
        inputs=[msg_input, image_input, chatbot, student_name_input],
        outputs=[chatbot, stats_display]
    ).then(lambda: '', outputs=[msg_input])

    practice_btn.click(generate_practice, inputs=[subject_input], outputs=[practice_output])
    reset_btn.click(reset_session_ui, outputs=[chatbot, stats_display, practice_output])

    gr.Markdown(
        '---\n'
        '**EduGemma** | Gemma 4 Good Hackathon 2026 | Future of Education Track\n'
        'Powered by `google/gemma-4-9b-it` + RAG + Adaptive Learning'
    )

demo.launch(share=True, debug=False)
print('🌐 Gradio demo running!')